In [1]:
!uv pip install gitsource fastembed

Resolved 37 packages in 3.73s                                        
Prepared 5 packages in 17.76s                                            
Installed 5 packages in 2ms                                 
 + fastembed==0.8.0
 + loguru==0.7.3
 + mmh3==5.2.1
 + pillow==12.2.0
 + py-rust-stemmers==0.1.8


In [2]:
!uv pip install gitsource --force-reinstall

Resolved 8 packages in 6.81s                                         
Prepared 8 packages in 0.38ms                                            
Uninstalled 8 packages in 4ms
Installed 8 packages in 2ms3.4.7                            
 ~ certifi==2026.6.17
 ~ charset-normalizer==3.4.7
 ~ gitsource==0.0.5
 ~ idna==3.18
 ~ python-frontmatter==1.3.0
 ~ pyyaml==6.0.3
 ~ requests==2.34.2
 ~ urllib3==2.7.0


In [4]:
%pip install gitsource fastembed

/home/usuario/Projects/llm-zoomcamp-lessons/llm-zoomcamp-onnx/.venv/bin/python3: No module named pip
Note: you may need to restart the kernel to use updated packages.


In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]
print(f"Loaded {len(documents)} pages successfully.")

Loaded 72 pages successfully.


In [2]:
from fastembed import TextEmbedding

# Initialize the model specified in the homework
embedding_model = TextEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")

query = "How does approximate nearest neighbor search work?"

# Extract the embedding vector
embeddings = list(embedding_model.embed([query]))
v = embeddings[0]

print(f"Vector Dimensions: {len(v)}")
print(f"v[0] Value: {v[0]:.2f}")

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Vector Dimensions: 384
v[0] Value: -0.02


In [4]:
# 1. Find the target document using dictionary key lookups
target_file = "02-vector-search/lessons/07-sqlitesearch-vector.md"
doc_content = None

for doc in documents:
    # Use key access instead of attribute access
    if target_file in doc['filename']:
        doc_content = doc['content']
        break

if doc_content is None:
    raise ValueError(f"Document {target_file} not found in the documents list.")

# 2. Embed the entire document content
doc_embeddings = list(embedding_model.embed([doc_content]))
v_doc = doc_embeddings[0]

# 3. Compute dot product 
import numpy as np

similarity = np.dot(v, v_doc)

print(f"Target Document Found: {target_file}")
print(f"Cosine Similarity with Query: {similarity:.2f}")

Target Document Found: 02-vector-search/lessons/07-sqlitesearch-vector.md
Cosine Similarity with Query: 0.36


In [5]:
import numpy as np
from gitsource import chunk_documents

# 1. Break the data down into 2000-character windows with a 1000-character overlap
chunks = chunk_documents(documents, size=2000, step=1000)
print(f"Total chunks created: {len(chunks)}")

# 2. Extract text contents from chunks
# Note: Checking structure; gitsource chunks are usually dicts or objects with a 'content' field.
# Let's verify by grabbing the first chunk's text representation.
sample_chunk = chunks[0]
chunk_texts = [
    c['content'] if isinstance(c, dict) else c.content 
    for c in chunks
]

# 3. Vectorize all chunks using batch execution
# fastembed.embed returns an iterator, list() forces execution
chunk_embeddings = list(embedding_model.embed(chunk_texts))

# 4. Stack vectors into a 2D NumPy Matrix X
X = np.array(chunk_embeddings)

# 5. Score the query vector 'v' against all rows via dot product
scores = X.dot(v)

# 6. Locate the index of the highest score
highest_score_idx = np.argmax(scores)
best_chunk = chunks[highest_score_idx]

# Extract filename regardless of dict or object structure
best_filename = best_chunk['filename'] if isinstance(best_chunk, dict) else best_chunk.filename

print(f"Highest Score: {scores[highest_score_idx]:.4f}")
print(f"Winning Chunk Index: {highest_score_idx}")
print(f"Belongs to File: {best_filename}")

Total chunks created: 295
Highest Score: 0.6489
Winning Chunk Index: 94
Belongs to File: 02-vector-search/lessons/07-sqlitesearch-vector.md


In [8]:
import numpy as np
from minsearch import VectorSearch

# 1. Initialize VectorSearch defining fields used for exact keyword matching/filtering
vs = VectorSearch(keyword_fields=["filename", "id"])

# 2. Extract standard dictionary payload formats from the chunks list
payloads = [
    {
        "id": chunk.get('id', f"chunk_{idx}"),
        "filename": chunk.get('filename', ''),
        "content": chunk.get('content', '')
    }
    for idx, chunk in enumerate(chunks)
]

# 3. Fit the bulk vector matrix X and payloads list simultaneously
vs.fit(X, payloads)

# 4. Embed the target query for Q4
query_q4 = "What metric do we use to evaluate a search engine?"
query_vector_q4 = list(embedding_model.embed([query_q4]))[0]

# 5. Run the vector search
# minsearch returns a list of dictionaries directly
results = vs.search(query_vector=query_vector_q4, num_results=1)

# 6. Output the definitive top path
if results:
    top_result = results[0]
    print(f"Top Result Filename: {top_result.get('filename')}")
else:
    print("Search query returned zero items.")

Top Result Filename: 04-evaluation/lessons/05-search-metrics.md


In [9]:
from minsearch import Index

# 1. Initialize and fit the standard Keyword Text Index
text_index = Index(text_fields=["content"], keyword_fields=["filename", "id"])
text_index.fit(payloads)

# 2. Define the new query text
query_q5 = "How do I store vectors in PostgreSQL?"

# 3. Execute Keyword Search (Top 5)
text_results = text_index.search(query=query_q5, num_results=5)
text_filenames = set(doc["filename"] for doc in text_results)

# 4. Execute Vector Search (Top 5)
query_vector_q5 = list(embedding_model.embed([query_q5]))[0]
vector_results = vs.search(query_vector=query_vector_q5, num_results=5)
vector_filenames = set(doc["filename"] for doc in vector_results)

# 5. Determine which files are in vector results but NOT in text results
vector_only_files = vector_filenames - text_filenames

print("--- Text Search Top 5 Files ---")
for f in text_filenames: print(f)

print("\n--- Vector Search Top 5 Files ---")
for f in vector_filenames: print(f)

print("\n--- Files in Vector but NOT in Text ---")
for f in vector_only_files:
    print(f)

--- Text Search Top 5 Files ---
02-vector-search/lessons/02-embeddings.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/01-intro.md

--- Vector Search Top 5 Files ---
02-vector-search/lessons/08-pgvector.md
03-orchestration/lessons/05-rag.md

--- Files in Vector but NOT in Text ---
02-vector-search/lessons/08-pgvector.md


In [10]:
# 1. Define the query for Q6
query_q6 = "How do I give the model access to tools?"

# 2. Run Text Search (Get all results to give RRF enough data to fuse)
text_results_q6 = text_index.search(query=query_q6, num_results=20)

# 3. Run Vector Search
query_vector_q6 = list(embedding_model.embed([query_q6]))[0]
vector_results_q6 = vs.search(query_vector=query_vector_q6, num_results=20)

# 4. Modified RRF function using 'id' instead of 'start' for chunk identification
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            # Using 'id' instead of 'start' to match our current schema
            key = (doc["filename"], doc["id"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

# 5. Execute Reciprocal Rank Fusion
fused_results = rrf([vector_results_q6, text_results_q6], k=60, num_results=5)

# 6. Output the top result
if fused_results:
    top_fused = fused_results[0]
    print(f"Top RRF Rank 1 Filename: {top_fused['filename']}")
else:
    print("No results returned from RRF.")

Top RRF Rank 1 Filename: 01-agentic-rag/lessons/13-function-calling.md
